# 验证数据管道中的Dataset、Loader

In [1]:
import os
import sys
from pathlib import Path

import torch


def find_project_root() -> Path:
    """Find the repository root independently of Jupyter's start directory."""
    override = os.environ.get("STEMNIST_PROJECT_ROOT")
    start = Path(override).expanduser() if override else Path.cwd()
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (candidate / "src" / "data" / "dataset.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter inside the repository "
        "or set STEMNIST_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()
project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

from src.data.dataset import STEMNISTDataset
from src.data.transform import build_pressure_transform


pressure_transform = build_pressure_transform()

pressure_dataset = STEMNISTDataset(
    data_root=PROJECT_ROOT / "data",
    split="train",
    data_kind="pressure",
    transform=pressure_transform,
)

inputs, target = pressure_dataset[0]

# 训练集应有 5390 个样本。
assert len(pressure_dataset) == 5390

# 单样本输出采用 [T, C, H, W]。
assert inputs.shape == (240, 1, 16, 16)

# 模型输入使用 float32。
assert inputs.dtype == torch.float32

# 分类标签使用 long。
assert target.dtype == torch.long

# 默认压力 transform 将数值缩放到 [0, 1]。
assert inputs.min().item() >= 0.0
assert inputs.max().item() <= 1.0

print("压力数据验证通过")

压力数据验证通过


In [3]:
spike_dataset = STEMNISTDataset(
    data_root=PROJECT_ROOT / "data",
    split="train",
    data_kind="spike",
)

spikes, spike_target = spike_dataset[0]

assert len(spike_dataset) == 5390
assert spikes.shape == (240, 1, 16, 16)
assert spikes.dtype == torch.float32

# 脉冲数据必须保持二值。
spike_values = set(
    torch.unique(spikes).tolist()
)

assert spike_values.issubset({0.0, 1.0})

print("脉冲数据验证通过")

脉冲数据验证通过


In [4]:
assert len(pressure_dataset) == len(spike_dataset)

for index in range(len(pressure_dataset)):
    pressure_info = pressure_dataset.sample_info(index)
    spike_info = spike_dataset.sample_info(index)

    assert pressure_info["sample_id"] == spike_info["sample_id"]
    assert pressure_info["label_index"] == spike_info["label_index"]

print("压力数据和脉冲数据完全对齐")

压力数据和脉冲数据完全对齐


In [6]:
from src.data.loader import LoaderConfig, create_loaders


config = LoaderConfig(
    batch_size=32,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    seed=42,
)

pressure_loaders = create_loaders(
    data_root=PROJECT_ROOT / "data",
    data_kind="pressure",
    train_transform=build_pressure_transform(),
    eval_transform=build_pressure_transform(),
    config=config,
)

batch_inputs, batch_targets = next(
    iter(pressure_loaders["train"])
)

assert batch_inputs.shape == (
    32,
    240,
    1,
    16,
    16,
)

assert batch_targets.shape == (32,)

print("压力 DataLoader 验证通过")

压力 DataLoader 验证通过


In [8]:
spike_loaders = create_loaders(
    data_root=PROJECT_ROOT / "data",
    data_kind="spike",
    config=config,
)

batch_spikes, batch_targets = next(
    iter(spike_loaders["train"])
)

assert batch_spikes.shape == (
    32,
    240,
    1,
    16,
    16,
)

print("脉冲 DataLoader 验证通过")

脉冲 DataLoader 验证通过
